In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from collections import Counter
import itertools
from itertools import chain
import math
import logging
import ast
from sklearn.impute import SimpleImputer
import sys

TIMEPOINTS = ["1", "2","3","4",'5']
MAD_THRESHOLD = 3
COMPLICATIONS = ["FGR", "HDP", "sPTB"]
PROPORTION_THRESHOLD = 0.3
SUPER_CANDIDATE_THRESHOLD = 3

In [49]:
# to correctly read lists/dicts from dataframe
# return: literal value of x, convert from string to list/dict if necessary
def safe_eval(x):
    if isinstance(x, str):
        return eval(x)
    else:
        return x

#calculate median and MAD for control samples
# return: dataframe with analyte_ID, tissue, datatype, timepoint, median, and MAD
def getStats(df, t, datatype, tissue):
    temp = pd.DataFrame(index=df.columns)
    temp["analyte_ID"] = df.columns
    temp["tissue"] = tissue
    temp["datatype"] = datatype
    temp["timepoint"] = t
    temp["control_median"] = df.median()
    temp["control_MAD"] = stats.median_abs_deviation(df)
    return temp

# combine batch-split files at timepoint t
# return: dataframe of all samples across all batches
def mergeBatches(batches, dir_input, t): # depends on a specific format + nomenclature for the cleaned files
    temp = pd.DataFrame()
    for b in batches:
        try:
            samples = pd.read_csv(dir_input + "/Samples_" + str(b) + "_" + str(t) + ".csv", index_col=0)
            temp = pd.concat([temp, samples])
        except:
            continue
    return temp

# extract and format control reference values
# return: dictionary with keys = timepoints and values = control reference median and MAD values
#         list of all control IDs (including timepoint suffix)
# output: control_reference_statistics_<tissue>_<timepoint>_<data_type>.csv
def controlRefStats(samples, dir_output, datatype, tissue):
    controlAll = {} # dict to return with keys = timepoints and values = control reference median and MAD values
    control_IDs = []
    for t in TIMEPOINTS:
        control_IDs.extend(list(samples[t].index))
        controlAll[t] = getStats(samples[t], t, datatype, tissue)
        controlAll[t].to_csv(dir_output + "/control_reference_statistics_" + tissue + "_" + str(t) + ".csv")
    with open(dir_output + '/control_IDs.txt', 'w') as f:
        for line in control_IDs:
            f.write(f"{line}\n")
    return controlAll, control_IDs

# calculate MAD scores based on timepoint
# return: dictionary where keys = analyte and values = MAD score for the analyate at that timepoint
def getMADscores(df, controlRef, t):
    scores_dict = {}
    for m in df.columns:
        if controlRef[t].loc[m, "control_MAD"] == 0:
            logging.info(m + " at timepoint " + str(t) + " has zero_variance/a MAD value of 0 and has been removed from downstream analyses")
        else:
            try:
                temp = (df[m] - controlRef[t].loc[m, "control_median"]) / controlRef[t].loc[m, "control_MAD"]
                scores_dict[m] = temp
            except:
                logging.warning("A issue has occured when calculating MAD score of " + m + " at timepoint " + str(t) + ": control_median = " + str(controlRef[t].loc[m, "control_median"]) + ", control_MAD = " + str(controlRef[t].loc[m, "control_MAD"]))
    scores = pd.DataFrame(scores_dict, index=df.index)
    return scores

# calculate MAD scores for all samples and analytes
# return: dictionary where keys = timepoints and values = MAD score matrices with group, subgroup, gestational age, and gestational age at sample collection per sample
# output: mad_scores_matrix_<tissue>_<timepoint>_<data_type>.csv
def MADscores(samples, dir_output, controlRef, tissue):
    scoreMatrix = {}
    for t in TIMEPOINTS:
        scoreMatrix[t] = getMADscores(samples[t], controlRef, t)
        scoreMatrix[t].to_csv(dir_output + "/mad_scores_matrix_" + tissue + "_" + str(t) + ".csv")
    return scoreMatrix

# flag MAD score > 3 or < -3
# return: dictionary of matrices by timepoint with 1 = elevated, -1 = decreased, 0 = outlier 
# output: outlier_flags_matrix_<tissue>_<timepoint>_<data_type>.csv
def flagOutliers(dir_output, scoreMatrix, tissue):
    outliers = {}
    for t in TIMEPOINTS:
        outliers[t] = scoreMatrix[t].map(lambda x: 1 if x > MAD_THRESHOLD else (-1 if x < -MAD_THRESHOLD else 0))
        outliers[t].to_csv(dir_output + "/outlier_flags_matrix_" + tissue + "_" + str(t) + ".csv")
    return outliers

# remove metadata from dataframe and save in a separate dictionary
# return: metadata dictionary of keys = timepoint, values = dataframe of sample ID, group, subgroup, gest age, and gest age at collection
#         sample dictionary of keys = timepoint, values = dataframe of batch normalized and log2 transformed metabolite expression        
def splitData(dir_input, batches):
    allMeta = {}
    allSamples = {}
    for t in TIMEPOINTS:
        temp = mergeBatches(batches, dir_input, t)
        meta = temp[["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"]]
        meta["Group"] = meta["Group"].replace("sptb", "sPTB")

        samples = temp.drop(columns=["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"])
        allMeta[t] = meta
        allSamples[t] = samples
    return allMeta, allSamples

# helper function for filterOutliers
# return: dictionary of indices of each patient in each timepoint dataframe
def t_to_p(outlierMatrix, patient_metadata):
    temp = {t: {} for t in TIMEPOINTS}
    for t in TIMEPOINTS:
        for idx in outlierMatrix[t].index:
            for p in patient_metadata.keys():
                if p in idx:
                    if p in temp[t].keys(): # if the value already exists
                        temp[t][p].append(idx)
                    else: # if the values doens't exist yet
                        temp[t][p] = [idx]
                    break # allows for multiple samples per patient per timepoint
    return temp

def extractFilteredOutliers(outlierMatrix, scoreMatrix, analyte, subject, t_to_p_index, bySample, group, subgroup): 

    total_timepoints = 0 # to count the number of timepoints the analyte is an outlier at for this patient
    outlier_values = [] # to store outlier values (-1 or 1)
    outlier_timepoints = [] # to store the timepoints at which the analyte is an outlier
    outlier_samples = {} # to store sample IDs where analyte is an outlier
    outlier_SampleGestAge = {} # to store gestational age of samples where analyte is an outlier
    outlier_mads = {} # to store MADs scores for outlier analytes

    # for every timepoint
    for t in TIMEPOINTS:

        # check if patient is in this timepoint
        if subject not in t_to_p_index[t]: 
            continue # move onto the next timepoint

        # get the list of sample ids for the subject at timepoint t, can be multiple
        idx = t_to_p_index[t][subject]

        try:
        # if the analyte is an outlier in any sample at this timepoint
            if any([abs(outlierMatrix[t].at[x, analyte]) > 0 for x in idx]):

                # add the outlier values of idx samples to outlier_values list, remove any 0 values
                outlier_values.extend([outlierMatrix[t].loc[x, analyte] for x in idx if outlierMatrix[t].loc[x, analyte] != 0])

                # if outlier values are not no longer in the same direction (not all the same sign)
                if len(np.unique(np.sign(outlier_values))) != 1: 
                    continue # move on to the next timepoint

                # add timpeoint to outlier_timepoints
                outlier_timepoints.append(t)
                
                # add sample Ids to the list of outlier samples
                outlier_samples[t] = idx

                # add sample gestational ages
                for sample in idx:
                    if t in outlier_SampleGestAge.keys():
                        outlier_SampleGestAge[t].append(bySample[sample]["SampleGestAge"])
                    else:
                        outlier_SampleGestAge[t] = [bySample[sample]["SampleGestAge"]]

                # add MAD scores to outlier_mads
                outlier_mads[t] = [scoreMatrix[t].loc[x, analyte] for x in idx]
                # add to total timepoints
                total_timepoints += 1

        except: # move onto the next timpoint if anything fails
            #logging.info(f"Warning: Failed to extract filtered outliers for {subject}: {analyte} at timepoint {t}.")
            print(f"Warning: Failed to extract filtered outliers for {subject}: {analyte} at timepoint {t}.")
            continue

    total_outlier_timepoints = len(set(outlier_timepoints))
    list_outlier_mads = list(chain.from_iterable((list(outlier_mads.values()))))
    
    if total_outlier_timepoints >= 2:
        direction = "elevated" if sum(outlier_values) > 0 else "decreased"
        return {"SubjectID": subject,
                "analyte_ID": analyte,
                "Group": group,
                "Subgroup": subgroup,
                "total_timepoints": total_timepoints,
                "outlier_timepoint_count": total_outlier_timepoints,
                "outlier_direction": direction,
                "outlier_timepoints": outlier_timepoints,
                "outlier_samples": outlier_samples,
                "outlier_SampleGestAge": outlier_SampleGestAge,
                "outlier_mad_scores": outlier_mads,
                "mean_outlier_mad": sum(list_outlier_mads)/len(list_outlier_mads)}
    return

# Filter for patient x analyte combinations that have >= 2 outlier samples and all outliers are directionally consistent (all elevated OR decreased)
# return: dataframe with rows = patient x analytes and columns = filtered outlier info
# output: filtered_outliers_<tissue>_<data_type>.csv
def filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, bySample, bySubject):
    # create list to store filtered outliers (outliers for a patient = in at least 2 samples)
    results = []

    # get all analytes with outlier results across all timepoints
    allAnalytes = set(chain.from_iterable([list(outlierMatrix[x].columns) for x in outlierMatrix.keys()]))


    # Pre-compute index mappings to avoid O(N) string matching in the inner loop
    # This creates a mapping of: timepoint -> {SubjectID: exact_index_name}
    t_to_p_index = t_to_p(outlierMatrix, bySubject)

    # for every subject + metadata in the subject-level dictionary
    for p, metadata in bySubject.items():
            
            # get group and subgroup for that subject
            group = metadata["Group"][0]
            subgroup = metadata["Subgroup"][0]

            # for every analyte
            for m in allAnalytes:
                # extract the filtered outlier information for the subject-analyte pair
                newRow = extractFilteredOutliers(outlierMatrix, scoreMatrix, m, p, t_to_p_index, bySample, group, subgroup)
                try:
                    if any(newRow.values()): # only add the new entry if info was actually extracted
                        results.append(newRow)
                except:
                    continue

    # convert the list of dict (results) into a dataframe
    filtered = pd.DataFrame(results)

    # save to csv 
    filtered.to_csv(dir_output + "/filtered_outliers_" + tissue + ".csv")

    # return the filtered dataframe
    return filtered, allAnalytes

# List 1: Most Prevelent
#   Goal: analytes elevated in the most complication patients
#   Steps:
#       1. Filter to complication samples (exclude controls)
#       2. For each analyte, count number of unique patients showing elevation
#       3. Calculate % complications affected = (n_patients / total complications in data for this tissue) * 100
#       4. Rank analytes by % complication affected (descending)
#       5. Select top 50 analytes
#   Include in output:
#       analyte_ID
#       n_patients_affected
#       percent_complications_affected
#       mean_outlier_timepoints_per_patient
#       complication_types_represented
#   Output: biomarker_most_prevalent_<tissue>.csv
def mostPrevalent(dir_output, persistentMatrix, meta, analytes, tissue, top, status):
    results = []
    complicationOnly = persistentMatrix.loc[persistentMatrix["Group"] != "Control",:]
    totalComplications = len(meta.loc[meta["Group"] != "Control",:].index)
    #for m in analytes:
    #    elevatedCounts = Counter(complicationOnly.loc[complicationOnly["analyte_ID"] == a,:]["group"])
    for m in analytes:
        mOnly = complicationOnly.loc[complicationOnly["analyte_ID"] == m,:]
        if len(mOnly.index) == 0:
            continue
        n_patients = len(mOnly.index)
        percentAffected = (n_patients / totalComplications) * 100
        results.append({"analyte_ID": m,
                        "n_patients_affected": n_patients,
                        "percent_complications_affected": percentAffected,
                        "mean_outlier_timepoints_per_patient": mOnly["outlier_timepoint_count"].sum() / len(mOnly.index),
                        "complication_types_represented": mOnly["Group"].str.upper().unique().tolist()})
    # only get the top n analytes 
    n = round(len(results)*(top/100))
    prevalent = pd.DataFrame(results).sort_values(by=["percent_complications_affected"], ascending=False).iloc[0:n,:]
    prevalent.to_csv(f"{dir_output}/biomarker_most_prevalent_{tissue}_{status}.csv")
    return prevalent
        

# List 2: Most Persistent
#   Goal: Analytes showing sustained elevation across pregnancy
#   Steps:
#       1. For each analyte (complication samples only):
#           Calculate average number of outlier timepoints per affected individual
#           Calculate average proportion: (outlier_timepoints / total_available_timepoints)
#       2. Filter to analytes affecting >= 5 patients
#       3. Rank by average proportion of timepoints (descending)
#       4. Select top 10% analytes
#   Inlcude in output:
#       Analyte_ID
#       n_patients_affected
#       mean_outlier_timepoints_per_patient_affected
#       mean_proportion_timepoints (mean outlier timeopints / available timepoints)
#       max_consecutive timepoints (longest stretch of consecutive outlier timepoints)
#   Output: biomarker_most_persistent_<tissue>.csv
def mostPersistent(dir_output, persistentMatrix, analytes, tissue, top, status):
    results = []
    complicationOnly = persistentMatrix.loc[persistentMatrix["Group"] != "Control",:]
    for m in analytes:
        mOnly = complicationOnly.loc[complicationOnly["analyte_ID"] == m,:]
        if len(mOnly.index) < 5:
            continue
        meanOutlierTP = mOnly["outlier_timepoint_count"].sum() / len(mOnly.index)
       #meanTP = mOnly["total_timepoints"].sum() / len(mOnly.index)
       # proportion = meanOutlierTP / meanTP
        proportion = meanOutlierTP / len(TIMEPOINTS)
        maxConsecutive = []
        for p in mOnly["SubjectID"]:
            raw = mOnly.loc[mOnly["SubjectID"] == p,:]["outlier_timepoints"].iloc[0]
            outlierTP = safe_eval(raw)

            outlierTPstring = "".join(outlierTP)
            mergedTP = "".join(TIMEPOINTS)
            if outlierTPstring in mergedTP:
                if len(outlierTPstring) > len(maxConsecutive):
                    maxConsecutive = outlierTP
        results.append({"analyte_ID": m,
                        "n_patients_affected": len(mOnly.index),
                        "mean_outlier_timepoints_per_patient_affected": meanOutlierTP,
                        "mean_proportion_timepoints": proportion,
                        "max_consecutive_timepoints": maxConsecutive})
    n = round(len(results)*(top/100))
    persistent = pd.DataFrame(results).sort_values(by=["mean_proportion_timepoints"], ascending=False).iloc[0:n,:]
    persistent.to_csv(f"{dir_output}/biomarker_most_persistent_{tissue}_{status}.csv")
    return persistent


# List 3: Early Warning
#   Goal: Analytes elevated at earlist available sample
#   Steps:
#       1. Define early sample as first sample collected, regardless of gestational bin
#       2. For each analyte in complication samples:
#           Count patients showing elevation at their earliest available sample
#       3. Filter to analytes elevated early in >= 10 patients
#       4. Rank by % of patients elevated at earliest timepoint
#   Include in output:
#       analyte_ID
#       n_patients_elevated_at_earliest
#       percent_elevated_at_earliest
#       mean_MAD_score_at_earliest
#   Output: biomarker_early_warning_<tissue>.csv
def earlyWarning(dir_output, filteredOutlierMatrix, analytes, tissue, status, top):
    results = []
    #complicationOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["Group"] != "Control",:] filteredOuliterMatrix is already complication only
    earlistT = [safe_eval(x)[0] for x in filteredOutlierMatrix["outlier_timepoints"]]
    earliestSamples = {filteredOutlierMatrix["SubjectID"].iloc[i]: safe_eval(filteredOutlierMatrix["outlier_samples"].iloc[i])[earlistT[i]][0] for i in range(len(earlistT))}
    for m in analytes:
        mOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["analyte_ID"] == m,:]
        if len(mOnly.index) == 0:
            continue
        mEarlistT = [safe_eval(x)[0] for x in mOnly["outlier_timepoints"]]
        mEarlistSamples = {mOnly["SubjectID"].iloc[i]: safe_eval(mOnly["outlier_samples"].iloc[i])[mEarlistT[i]][0] for i in range(len(mEarlistT))}
        earliestMask = [mEarlistSamples[x] == earliestSamples[x] for x in mEarlistSamples.keys()]
        if sum(earliestMask) == 0:
            continue

        earliestMADscores = [safe_eval(mOnly.loc[earliestMask,"outlier_mad_scores"].iloc[i])[np.array(mEarlistT)[earliestMask].tolist()[i]][0] for i in range(sum(earliestMask))]

        results.append({"analyte_ID": m,
                        f"n_patients_{status}_at_earliest": sum(earliestMask),
                        f"percent_{status}_at_earliest": sum(earliestMask) / len(mOnly["SubjectID"]),
                        "mean_MAD_score_at_earliest": sum(earliestMADscores) / len(earliestMADscores)})
    n = round(len(results)*(top/100))
    warning = pd.DataFrame(results).sort_values(by=[f'n_patients_{status}_at_earliest'], ascending=False).iloc[0:n,:]
    warning.to_csv(f"{dir_output}/biomarker_early_warning_{tissue}_{status}.csv")
    return warning


# helper function for complicationSpecific biomarker analysis -> runs chi2 test
# return: if the pvalue is significant, dictionary of test results

def chi2_test(data, comparison, analyte, column, t, p_threshold, status, numComparisons):
    cross = pd.crosstab(data[column], # generate contingency matrix
                        data[analyte], 
                        margins = False)
    
    proportion0 = cross.loc[comparison[0],:]/sum(cross.loc[comparison[0],:])
    proportion1 = cross.loc[comparison[1],:]/sum(cross.loc[comparison[1],:])

    try:
        portion = proportion0[1]  # check if percentage of outliers in complication is less than the threshold
    except:
        portion = proportion0[-1]
    
    if portion < PROPORTION_THRESHOLD: # check if percentage of outliers in complication is less than the threshold
        return

    # scipy chisquared goes off of proportations
    test = stats.chisquare(proportion0, proportion1)
    #if test.pvalue <= p_threshold: # nan is not less than any number, should filter nan out
    if status == "elevated":
        return {"group": comparison[0],
                "reference": comparison[1],
                "analyte": analyte,
                "gestational_bin": t, 
                "outlier_count_in_group": cross.loc[comparison[0]][1],
                "total_count_in_group": sum(cross.loc[comparison[0]]),
                "outlier_count_in_reference": cross.loc[comparison[1]][1],
                "total_count_in_reference": sum(cross.loc[comparison[1]]),
                "outlier_percentage_in_group": proportion0[1],
                "outlier_percentage_in_reference": proportion1[1], 
                "chi2_statistic": test.statistic,
                "chi2_pvalue": test.pvalue,
                "chi2_adjPvalue": 1.0 if test.pvalue*numComparisons>=1.0 else test.pvalue*numComparisons} # bonferonni correction, multiply p-value by the number of comparisons being done
    else:
        return {"group": comparison[0],
                "reference": comparison[1],
                "analyte": analyte,
                "gestational_bin": t, 
                "outlier_count_in_group": cross.loc[comparison[0]][-1],
                "total_count_in_group": sum(cross.loc[comparison[0]]),
                "outlier_count_in_reference": cross.loc[comparison[1]][-1],
                "total_count_in_reference": sum(cross.loc[comparison[1]]),
                "outlier_percentage_in_group": proportion0[-1],
                "outlier_percentage_in_reference": proportion1[-1], 
                "chi2_statistic": test.statistic,
                "chi2_pvalue": test.pvalue,
                "chi2_adjPvalue": 1.0 if test.pvalue*numComparisons>=1.0 else test.pvalue*numComparisons} 

#   Goal: Analytes enriched in specific complication subtypes by all timepoints + individually
#   Steps:
#       1. For each complication type (FGR, HDP, sPTB) separately (and all together)
#           Calculate % of that complication type showing each analyte elevated
#       2. For each analyte:
#           calculate chi2 statistic + p-value
#       3. Filter to analytes with:
#           30% prevelence in at least one complication type
#           p_value < threshold (default 0.05)
#       4. Rank by p_value (ascending)
#   Include in output:
#       analyte_ID
#       primary_complication_type
#       percent_in_primary_complication
#       percent_in_other_complications
#       chi2_value
#       p_value
#       n_patients_primary_complication
#   Output: biomarker_complication_specific_<tissue>.csv
def complicationSpecific(t, bySample, allAnalytes, tissue, dir_output, p_threshold = 0.05, mode = ["all", "specific"], status = ["elevated", "decreased"]):
    # given list of COMPLCIATIONS + Controls, run all chi-squared tests
    results = []

    if t not in TIMEPOINTS: # check if gestational bin is valid, only allowing for timepint specific analysis, not all together
        logging.warning(f"{t} is not a valid gestational bin.")

    # get the outlier file for the timepoint
    outliers_t = pd.read_csv(f"{dir_output}/outlier_flags_matrix_plasma_{t}.csv")

    # double check there's metadata for the sample and add the group info
    if sum([x not in bySample.keys() for x in outliers_t["SampleID"]]) > 0:
        print("Some samples missing in bySample?")
        print(outliers_t.loc[[x not in bySample.keys() for x in outliers_t["SampleID"]], "SampleID"])

    outliers_t = outliers_t.loc[[x in bySample.keys() for x in outliers_t["SampleID"]], :]
    outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
    
    if mode == "all": # if we're comparing all complications to control
        comparisons = [("Complication", "Control")]
        outliers_t["AllComplication"] = ["Control" if x == "Control" else "Complication" for x in outliers_t["Group"]]
        column = "AllComplication"
    else: # if we're comparing by specific complications to each other + control
        comparisons = list(itertools.permutations(COMPLICATIONS,2)) + [(x, "Control") for x in COMPLICATIONS]
        column = "Group"
    
    for m in allAnalytes: # for every analyte
        try:
            for c in comparisons: # for every comparison
                if status == "elevated": # if we're only looking at elevated analytes in complications
                    temp = outliers_t.loc[[x >= 0 for x in outliers_t[m]],:]
                else:
                    temp = outliers_t.loc[[x <= 0 for x in outliers_t[m]],:]
                test = chi2_test(temp, c, m, column, t, p_threshold, status, len(comparisons))

                if test: # if the test was significant/returned a value
                    results.append(test)
        except:
            logging.info(f"Analyte {m} was not included in {t} gestational bin outlier analysis - excluded from complication-specific chi-squared testing.")
         
        
    if results:
        specific = pd.DataFrame(results).sort_values(by="chi2_adjPvalue", ascending=True) # ascending=True so smallest p-values at the top
        specific.to_csv(f"{dir_output}/biomarker_complication_specific_{tissue}_{t}_{mode}_{status}.csv")
        significant = specific.loc[specific["chi2_adjPvalue"] < p_threshold,:]
        significant.to_csv(f"{dir_output}/biomarker_complication_specific_{tissue}_{t}_{mode}_{status}_{str(p_threshold)}.csv")
        return significant
    return


        
# List 5: Most Extreme
#   Goal: Analytes with highest magnitude deviations at all timepoints + individually
#   Steps:
#       1. For each analyte (complications only):
#           Calculate median MAD score across all outlier instances
#           Calculate 99th percentile MAD score
#           Calculate max MAD score observed
#       2. Filter to analytes affecting >=5 patients
#       3. Rank by median MAD score (descending)
#       4. Select top 10% analytes
#   Include in output:
#       analyte_ID
#       n_patients_affected
#       median_MAD_score
#       percentile_99_MAD_score
#       max_MAD_score
#       patient_with_max (SubjectID showing maximum deviation)
#   Output: biomarker_most_extreme_<tissue>.csv
def mostExtreme(dir_output, filteredOutlierMatrix, analytes, tissue, top, gestationalBin, status = ["elevated", "decreased"]):
    results = []

    if gestationalBin not in TIMEPOINTS:
        logging.error(f"Invalid gestational bin {gestationalBin} for mostExtreme biomarker analysis.")
        return
    
    binOnly = filteredOutlierMatrix[[gestationalBin in x for x in filteredOutlierMatrix["outlier_timepoints"]]]

    for m in analytes:
        mOnly = binOnly.loc[binOnly["analyte_ID"] == m,:]
        all_mad_scores = list(chain.from_iterable([safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]))
        if len(all_mad_scores) < 5:
            continue
        if status == "elevated":
            order = False
            results.append({"analyte_ID": m,
                            "n_patients_affected": len(mOnly.index),
                            "median_MAD_score": np.median(all_mad_scores),
                            "percentile_99_MAD_score": np.percentile(all_mad_scores, 99),
                            "max_MAD_score": max(all_mad_scores),
                            "patient_with_max": list(mOnly["SubjectID"][[max(all_mad_scores) in x for x in [safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]]])
                            })
        else:
            order = True
            results.append({"analyte_ID": m,
                            "n_patients_affected": len(mOnly.index),
                            "median_MAD_score": np.median(all_mad_scores),
                            "percentile_99_MAD_score": np.percentile(all_mad_scores, 99),
                            "min_MAD_score": min(all_mad_scores),
                            "patient_with_min": list(mOnly["SubjectID"][[min(all_mad_scores) in x for x in [safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]]])
                            })
    n = round(len(results)*(top/100))
    extreme = pd.DataFrame(results).sort_values(by="median_MAD_score", ascending=order).iloc[0:n,:]
    extreme.to_csv(f"{dir_output}/biomarker_most_extreme_{tissue}_{gestationalBin}_{status}.csv")
    return extreme

# helper function for running all biomarker identification functions
def identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = True):
    if elevated:
        subset = filteredOutlierMatrix.loc[filteredOutlierMatrix["outlier_direction"] == "elevated",:]
        status = "elevated"
    else:
        subset = filteredOutlierMatrix.loc[filteredOutlierMatrix["outlier_direction"] == "decreased",:]
        status = "decreased"

    mergedMeta = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True).drop_duplicates(subset=["SubjectID"])

    prevalentMarkers = mostPrevalent(dir_output, subset, mergedMeta, allAnalytes, tissue, top, status)
    persistentMarkers = mostPersistent(dir_output, subset, allAnalytes, tissue, top, status)
    earlyMarkers = earlyWarning(dir_output, subset, allAnalytes, tissue, status, top)

    specificMarkers_specific = {}
    #specificMarkers_all = {}
    extremeMarkers = {}
    for t in TIMEPOINTS:
        specificMarkers_specific[t] = complicationSpecific(t, bySample, allAnalytes, tissue, dir_output, p_threshold = 0.05, mode =  "specific", status = status)
        #specificMarkers_all[t] = complicationSpecific(t, bySample, allAnalytes, "plasma", dir_output, p_threshold = 0.05, mode =  "all", status = "decreased")
        extremeMarkers[t] = mostExtreme(dir_output, subset, allAnalytes, tissue, top, t, status)

    return prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers_specific, extremeMarkers

# generate crosswalk matrix
# return: dataframe where rows = metabolites, columns = category of outlier, values = 1 if in category, 0 if not
def crosswalkMatrix(dir_output, analytes, prevalent, persistent, early, specific, extreme, tissue, status):
    df = []
    for m in analytes:
        newRow = {}
        total = 0
        newRow["analyte_ID"] = m
        if m in set(prevalent["analyte_ID"]):
            total += 1
            entry = {}
            for column in prevalent.columns:
                if column != "analyte_ID" and column != 'Unnamed: 0':
                    entry[column]= list(prevalent.loc[prevalent["analyte_ID" ] == m, column])[0]
            newRow["most_prevalent"] = entry

        if m in set(persistent["analyte_ID"]):
            total += 1
            newRow["most_persistent"] = 1
            entry = {}
            for column in persistent.columns:
                if column != "analyte_ID" and column != 'Unnamed: 0':
                    entry[column]= list(persistent.loc[persistent["analyte_ID" ] == m, column])[0]
            newRow["most_persistent"] = entry

        if m in set(early["analyte_ID"]):
            total += 1
            newRow["early_warning"] = 1
            entry = {}
            for column in early.columns:
                if column != "analyte_ID" and column != 'Unnamed: 0':
                    entry[column]= list(early.loc[early["analyte_ID" ] == m, column])[0]
            newRow["early_warning"] = entry

        inSpecific = False
        inExtreme = False
        for t in TIMEPOINTS:
            temp = specific[t]
            if m in set(temp["analyte"]):
                inSpecific = True
                newRow[f"complication_specific_{t}"] = 1
                entry = {}
                for column in temp.columns:
                    if column != "analyte" and column != 'Unnamed: 0':
                        entry[column]= list(temp.loc[temp["analyte" ] == m, column])[0]
                newRow[f"complication_specific_{t}"] = entry

            temp = extreme[t]
            if m in set(temp["analyte_ID"]):
                inExtreme = True
                newRow[f"most_extreme_{t}"] = 1
                entry = {}
                for column in temp.columns:
                    if column != "analyte_ID" and column != 'Unnamed: 0':
                        entry[column]= list(temp.loc[temp["analyte_ID" ] == m, column])[0]
                newRow[f"most_extreme_{t}"] = entry
                
        if inSpecific:
            total += 1
        if inExtreme:
            total += 1

        newRow["total"] = total
        df.append(newRow)

    df = pd.DataFrame(df)
    super_candidates = df[df["total"] >= SUPER_CANDIDATE_THRESHOLD]
    logging.info(f"{str(len(super_candidates.index))} super candidate metabolites (in >={SUPER_CANDIDATE_THRESHOLD} lists) identified.")
    df.to_csv(f"{dir_output}/biomarker_summary_crosswalk_{tissue}_{status}.csv")
    super_candidates.to_csv(f"{dir_output}/biomarker_summary_super_candidates_{tissue}_{status}.csv")
    return df


    
# primary wrapper function for Outlier Analysis
def OutlierAnalysis(dir_input, dir_output, datatype, tissue, batches, top):
    meta, samples = splitData(dir_input, batches)

    unique_samples = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True)
    unique_samples = unique_samples[unique_samples["Group"] != "Control"]
    bySample = unique_samples.set_index("SampleID")[["SubjectID", "Group", "Subgroup", "SampleGestAge", "Timepoint"]].to_dict('index')

    # get subject-level information -> dict
    bySubject = unique_samples.groupby('SubjectID').agg(lambda x: x.unique().tolist()).reset_index().set_index("SubjectID").to_dict('index')

    logging.info("Calculating control reference statistics...")
    controlRef, control_IDs = controlRefStats(samples, dir_output, datatype, tissue)
    logging.info("Calculating sample MAD scores...")
    scoreMatrix = MADscores(samples, dir_output, controlRef, tissue, datatype)
    logging.info("Flagging outliers by patient x analyte across timepoints...")
    outlierMatrix = flagOutliers(dir_output, scoreMatrix, tissue, datatype)
    logging.info("Identifying persistent and consistent outliers...")
    filteredOutlierMatrix, allAnalytes = filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, datatype, bySample, bySubject)
    #filteredOutlierMatrix = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/filtered_outliers_plasma_PROT.csv")

    # identify elevated biomarkers
    prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = True)
    crosswalkMatrix(dir_output, allAnalytes, prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers, tissue, status)

    # identify decreased biomarkers
    prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = False)
    crosswalkMatrix(dir_output, allAnalytes, prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers, tissue, status)
    return

def main():
    dir_input = sys.argv[1] # e.g. /Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma
    dir_output = sys.argv[2] # e.g. /Users/kaylaxu/Desktop/data/MAD_analyses

    batches = pd.read_csv(dir_input + "/pos_batch.csv")["batch"].unique().tolist()

    if "MTBL" in dir_input:
        datatype = "MTBL"
    elif "LIPD" in dir_input:
        datatype = "LIPD"
    elif "PROT" in dir_input:
        datatype = "PROT"

    if "plasma" in dir_input:
        tissue = "plasma"
    else:
        tissue = "placenta"

    logging.basicConfig( # initiate log file
        filename= datatype + '_outlierAnalysis.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        filemode='w'  # Use 'w' to overwrite the file each run, or 'a' to append
    )
    logging.info("Initializing " + datatype + " MAD outlier analysis...")

    OutlierAnalysis(dir_input, dir_output, datatype, tissue, batches, top=10) 

    logging.info("DONE - " + datatype + " MAD outlier analysis complete!")
    #close log file
    logging.shutdown()
    return

Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
['DP3-0093A', 'DP3-0273A', 'DP3-0364A']
Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
[]
Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
['DP3-0358EC']
Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
['DP3-0156D']
Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
['DP3-0272D']


Dropped analytes for >50% missing: 
[]
Dropped samples for >20% missing: 
['DP3-0018', 'DP3-0056', 'DP3-0111', 'DP3-0313', 'DP3-0420']
Total Samples: 108
Num. of Unique Subjects: 108
Total Features = 991


In [4]:
dir_output = "/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/MTBL"
datatype = "MTBL"
tissue = "plasma"
top = 10


logging.basicConfig( # initiate log file
        filename= datatype + '_outlierAnalysis.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        filemode='w'  # Use 'w' to overwrite the file each run, or 'a' to append
    )
logging.info("Initializing " + datatype + " MAD outlier analysis...")

logging.info("Calculating control reference statistics...")
controlRef, control_IDs = controlRefStats(samples, dir_output, datatype, tissue)
logging.info("Calculating sample MAD scores...")
scoreMatrix = MADscores(samples, dir_output, controlRef, tissue)
logging.info("Flagging outliers by patient x analyte across timepoints...")
outlierMatrix = flagOutliers(dir_output, scoreMatrix, tissue)
logging.info("Identifying persistent and consistent outliers...")
filteredOutlierMatrix, allAnalytes = filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, bySample, bySubject)
#filteredOutlierMatrix = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/filtered_outliers_plasma_PROT.csv")
    # identify biomarkers
#prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes=allAnalytes, top=top)
#elevatedOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["outlier_direction"] == "elevated",:]

In [5]:
prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = True)
crosswalkMatrix(dir_output, allAnalytes, prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers, tissue, status = "elevated")


Some samples missing in bySample?
1       DP3-0006A
2       DP3-0008A
4       DP3-0018A
8       DP3-0027A
9       DP3-0028A
10     DP3-0029EA
11      DP3-0034A
13      DP3-0037A
19      DP3-0057A
20      DP3-0062A
22      DP3-0070A
24      DP3-0075A
28      DP3-0094A
30      DP3-0105A
32      DP3-0111A
33      DP3-0112A
34      DP3-0113A
35      DP3-0116A
36      DP3-0120A
37      DP3-0121A
39      DP3-0126A
40      DP3-0129A
41     DP3-0130EA
44      DP3-0142A
45      DP3-0152A
52      DP3-0166A
53      DP3-0169A
54      DP3-0175A
55      DP3-0177A
56      DP3-0179A
57      DP3-0181A
60      DP3-0185A
61      DP3-0200A
62      DP3-0202A
63      DP3-0204A
66      DP3-0217A
70      DP3-0244A
72      DP3-0254A
73      DP3-0256A
79      DP3-0272A
80      DP3-0277A
81      DP3-0279A
83      DP3-0284A
84      DP3-0287A
85      DP3-0292A
86      DP3-0296A
88      DP3-0300A
93      DP3-0313A
101     DP3-0346A
103     DP3-0353A
111     DP3-0381A
112     DP3-0387A
113     DP3-0388A
119     DP3-

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
1       DP3-0006B
3       DP3-0018B
5       DP3-0027B
6       DP3-0028B
7      DP3-0029EB
12      DP3-0057B
14      DP3-0070B
17      DP3-0075B
21      DP3-0093B
22      DP3-0094B
24      DP3-0105B
25      DP3-0112B
26      DP3-0113B
27      DP3-0116B
28      DP3-0120B
29      DP3-0121B
31      DP3-0126B
32      DP3-0129B
33     DP3-0130EB
36      DP3-0142B
42      DP3-0166B
43      DP3-0169B
44      DP3-0175B
45      DP3-0177B
46      DP3-0179B
47      DP3-0181B
49      DP3-0185B
52      DP3-0202B
55      DP3-0217B
58      DP3-0244B
62      DP3-0254B
63      DP3-0256B
66      DP3-0272B
67      DP3-0273B
68      DP3-0277B
69      DP3-0279B
71      DP3-0284B
72      DP3-0287B
73      DP3-0296B
77      DP3-0313B
92      DP3-0381B
93      DP3-0387B
94      DP3-0388B
101     DP3-0416B
Name: SampleID, dtype: str


/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
1       DP3-0008B
5       DP3-0027C
6       DP3-0034B
7       DP3-0037B
8       DP3-0037C
15      DP3-0057C
16      DP3-0062B
18      DP3-0070C
21      DP3-0075C
23      DP3-0093C
24      DP3-0094C
26      DP3-0105C
29      DP3-0111B
30      DP3-0111C
31      DP3-0112C
32      DP3-0113C
33      DP3-0120C
34      DP3-0121C
35      DP3-0126C
36      DP3-0129C
37     DP3-0130EC
40      DP3-0142C
41      DP3-0152B
48      DP3-0166C
49      DP3-0169C
50      DP3-0175C
51      DP3-0177C
52      DP3-0179C
53      DP3-0181C
57      DP3-0185C
59      DP3-0200B
60      DP3-0200C
63      DP3-0202C
64      DP3-0204B
65      DP3-0204C
70      DP3-0217C
74      DP3-0244C
77      DP3-0254C
78      DP3-0256C
85      DP3-0272C
86      DP3-0273C
87      DP3-0277C
88      DP3-0279C
90      DP3-0284C
91      DP3-0287C
92      DP3-0292B
93      DP3-0292C
94      DP3-0296C
96      DP3-0300B
97      DP3-0300C
102     DP3-0313C
112     DP3-0346B
113     DP3-0346C
115     DP3-

...

Some samples missing in bySample?
1      DP3-0006C
2      DP3-0008C
3      DP3-0018C
5      DP3-0028C
6     DP3-0029EC
7      DP3-0034C
11     DP3-0057D
12     DP3-0062C
14     DP3-0094D
15     DP3-0112D
16     DP3-0116C
17     DP3-0121D
19    DP3-0130ED
21     DP3-0152C
28     DP3-0254D
31     DP3-0273D
32     DP3-0279D
47     DP3-0381D
48     DP3-0387D
Name: SampleID, dtype: str


/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
0      DP3-0006D
1      DP3-0008D
3      DP3-0018D
5      DP3-0027D
6      DP3-0028D
7     DP3-0029ED
8      DP3-0034D
11     DP3-0037D
13     DP3-0062D
15     DP3-0070D
18     DP3-0075D
21     DP3-0093D
22     DP3-0105D
24     DP3-0111D
25     DP3-0113D
26     DP3-0116D
27     DP3-0120D
28     DP3-0126D
29     DP3-0129D
30     DP3-0142D
31     DP3-0152D
35     DP3-0166D
36     DP3-0169D
37     DP3-0175D
38     DP3-0177D
39     DP3-0179D
40     DP3-0181D
43     DP3-0185D
44     DP3-0200D
45     DP3-0202D
46     DP3-0204D
47     DP3-0217D
48     DP3-0217E
51     DP3-0244D
52     DP3-0244E
55     DP3-0256D
56     DP3-0256E
58     DP3-0273E
59     DP3-0277D
60     DP3-0277E
61     DP3-0279E
62     DP3-0284D
63     DP3-0287D
64     DP3-0287E
65     DP3-0292D
66     DP3-0296D
68     DP3-0300D
70     DP3-0313D
71     DP3-0313E
73     DP3-0346D
75     DP3-0353D
83     DP3-0381E
84     DP3-0387E
85     DP3-0388D
86     DP3-0388E
91     DP3-0416D
Name: SampleID

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp


,analyte_ID,total,complication_specific_4,most_prevalent,early_warning,most_extreme_3,complication_specific_2,complication_specific_1,most_extreme_5,most_persistent,most_extreme_2,most_extreme_1,most_extreme_4,complication_specific_3,complication_specific_5
0,Metronidazole_POS,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,unk_n1621_NEG_289.0694_RT3.38,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,L-Serine_NEG,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gaburedin B_POS,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Creatine_POS,1,"{'group': 'sPTB', 'reference': 'HDP', 'gestati...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,unk_n101_NEG_80.9748_RT3.00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
534,N-Desmethyltramadol_POS,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
535,(S)-3-[(Cyanophenylmethyl)amino]-3-oxopropanoi...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536,alpha-Ketoisovaleric acid_NEG,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:

# identify decreased biomarkers
prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = False)
crosswalkMatrix(dir_output, allAnalytes, prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers, tissue, status="decreased")

Some samples missing in bySample?
1       DP3-0006A
2       DP3-0008A
4       DP3-0018A
8       DP3-0027A
9       DP3-0028A
10     DP3-0029EA
11      DP3-0034A
13      DP3-0037A
19      DP3-0057A
20      DP3-0062A
22      DP3-0070A
24      DP3-0075A
28      DP3-0094A
30      DP3-0105A
32      DP3-0111A
33      DP3-0112A
34      DP3-0113A
35      DP3-0116A
36      DP3-0120A
37      DP3-0121A
39      DP3-0126A
40      DP3-0129A
41     DP3-0130EA
44      DP3-0142A
45      DP3-0152A
52      DP3-0166A
53      DP3-0169A
54      DP3-0175A
55      DP3-0177A
56      DP3-0179A
57      DP3-0181A
60      DP3-0185A
61      DP3-0200A
62      DP3-0202A
63      DP3-0204A
66      DP3-0217A
70      DP3-0244A
72      DP3-0254A
73      DP3-0256A
79      DP3-0272A
80      DP3-0277A
81      DP3-0279A
83      DP3-0284A
84      DP3-0287A
85      DP3-0292A
86      DP3-0296A
88      DP3-0300A
93      DP3-0313A
101     DP3-0346A
103     DP3-0353A
111     DP3-0381A
112     DP3-0387A
113     DP3-0388A
119     DP3-

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
1       DP3-0006B
3       DP3-0018B
5       DP3-0027B
6       DP3-0028B
7      DP3-0029EB
12      DP3-0057B
14      DP3-0070B
17      DP3-0075B
21      DP3-0093B
22      DP3-0094B
24      DP3-0105B
25      DP3-0112B
26      DP3-0113B
27      DP3-0116B
28      DP3-0120B
29      DP3-0121B
31      DP3-0126B
32      DP3-0129B
33     DP3-0130EB
36      DP3-0142B
42      DP3-0166B
43      DP3-0169B
44      DP3-0175B
45      DP3-0177B
46      DP3-0179B
47      DP3-0181B
49      DP3-0185B
52      DP3-0202B
55      DP3-0217B
58      DP3-0244B
62      DP3-0254B
63      DP3-0256B
66      DP3-0272B
67      DP3-0273B
68      DP3-0277B
69      DP3-0279B
71      DP3-0284B
72      DP3-0287B
73      DP3-0296B
77      DP3-0313B
92      DP3-0381B
93      DP3-0387B
94      DP3-0388B
101     DP3-0416B
Name: SampleID, dtype: str


/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
1       DP3-0008B
5       DP3-0027C
6       DP3-0034B
7       DP3-0037B
8       DP3-0037C
15      DP3-0057C
16      DP3-0062B
18      DP3-0070C
21      DP3-0075C
23      DP3-0093C
24      DP3-0094C
26      DP3-0105C
29      DP3-0111B
30      DP3-0111C
31      DP3-0112C
32      DP3-0113C
33      DP3-0120C
34      DP3-0121C
35      DP3-0126C
36      DP3-0129C
37     DP3-0130EC
40      DP3-0142C
41      DP3-0152B
48      DP3-0166C
49      DP3-0169C
50      DP3-0175C
51      DP3-0177C
52      DP3-0179C
53      DP3-0181C
57      DP3-0185C
59      DP3-0200B
60      DP3-0200C
63      DP3-0202C
64      DP3-0204B
65      DP3-0204C
70      DP3-0217C
74      DP3-0244C
77      DP3-0254C
78      DP3-0256C
85      DP3-0272C
86      DP3-0273C
87      DP3-0277C
88      DP3-0279C
90      DP3-0284C
91      DP3-0287C
92      DP3-0292B
93      DP3-0292C
94      DP3-0296C
96      DP3-0300B
97      DP3-0300C
102     DP3-0313C
112     DP3-0346B
113     DP3-0346C
115     DP3-

...

Some samples missing in bySample?
1      DP3-0006C
2      DP3-0008C
3      DP3-0018C
5      DP3-0028C
6     DP3-0029EC
7      DP3-0034C
11     DP3-0057D
12     DP3-0062C
14     DP3-0094D
15     DP3-0112D
16     DP3-0116C
17     DP3-0121D
19    DP3-0130ED
21     DP3-0152C
28     DP3-0254D
31     DP3-0273D
32     DP3-0279D
47     DP3-0381D
48     DP3-0387D
Name: SampleID, dtype: str


/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in d

Some samples missing in bySample?
0      DP3-0006D
1      DP3-0008D
3      DP3-0018D
5      DP3-0027D
6      DP3-0028D
7     DP3-0029ED
8      DP3-0034D
11     DP3-0037D
13     DP3-0062D
15     DP3-0070D
18     DP3-0075D
21     DP3-0093D
22     DP3-0105D
24     DP3-0111D
25     DP3-0113D
26     DP3-0116D
27     DP3-0120D
28     DP3-0126D
29     DP3-0129D
30     DP3-0142D
31     DP3-0152D
35     DP3-0166D
36     DP3-0169D
37     DP3-0175D
38     DP3-0177D
39     DP3-0179D
40     DP3-0181D
43     DP3-0185D
44     DP3-0200D
45     DP3-0202D
46     DP3-0204D
47     DP3-0217D
48     DP3-0217E
51     DP3-0244D
52     DP3-0244E
55     DP3-0256D
56     DP3-0256E
58     DP3-0273E
59     DP3-0277D
60     DP3-0277E
61     DP3-0279E
62     DP3-0284D
63     DP3-0287D
64     DP3-0287E
65     DP3-0292D
66     DP3-0296D
68     DP3-0300D
70     DP3-0313D
71     DP3-0313E
73     DP3-0346D
75     DP3-0353D
83     DP3-0381E
84     DP3-0387E
85     DP3-0388D
86     DP3-0388E
91     DP3-0416D
Name: SampleID

/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_86013/1195996763.py:443: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]


,analyte_ID,early_warning,complication_specific_1,complication_specific_2,complication_specific_3,total,complication_specific_4,most_prevalent,most_extreme_5,most_persistent,most_extreme_1,most_extreme_3,most_extreme_2,most_extreme_4
0,Metronidazole_POS,"{'n_patients_decreased_at_earliest': 7, 'perce...","{'group': 'sPTB', 'reference': 'HDP', 'gestati...","{'group': 'sPTB', 'reference': 'HDP', 'gestati...","{'group': 'sPTB', 'reference': 'HDP', 'gestati...",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,unk_n1621_NEG_289.0694_RT3.38,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,L-Serine_NEG,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Gaburedin B_POS,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Creatine_POS,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533,unk_n101_NEG_80.9748_RT3.00,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
534,N-Desmethyltramadol_POS,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
535,(S)-3-[(Cyanophenylmethyl)amino]-3-oxopropanoi...,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536,alpha-Ketoisovaleric acid_NEG,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
totalSamples = 0
features = []
for i in meta.keys():
    if i == "1":
        time = "< 13.9 weeks"
    elif i == "2":
        time = "14 to 21.9 weeks"
    elif i == "3":
        time = "22 to 31.9 weeks"
    elif i == "4":
        time = "32 to 36.9 weeks"
    else:
        time = "> 37 weeks"
    print(f"Gestational Bin {i} ({time}): ")
    print(f"\t Total Samples: {len(meta[i].index)}")
    print(f"\t Num. of Unique Subjects: {len(meta[i]["SubjectID"].unique())}")
    totalSamples += len(meta[i].index)
    features.extend(samples[i].columns)
print(f"Total Samples = {totalSamples}")
print(f"Total Features = {len(list(set(features)))}")


Gestational Bin 1 (< 13.9 weeks): 
	 Total Samples: 123
	 Num. of Unique Subjects: 122
Gestational Bin 2 (14 to 21.9 weeks): 
	 Total Samples: 104
	 Num. of Unique Subjects: 102
Gestational Bin 3 (22 to 31.9 weeks): 
	 Total Samples: 147
	 Num. of Unique Subjects: 121
Gestational Bin 4 (32 to 36.9 weeks): 
	 Total Samples: 54
	 Num. of Unique Subjects: 52
Gestational Bin 5 (> 37 weeks): 
	 Total Samples: 96
	 Num. of Unique Subjects: 81
Total Samples = 524
Total Features = 538


In [ ]:
meta_cols = ["SampleID", "SubjectID", "Batch", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint", "SampleTimepoint",
             "MetadataCanonicalID", "labor_onset", "indicated_onset", "spont_labor_flag", "indicated_onset_flag", "cat_labor_onset_flag",
             "within_0_1wk_delivery_flag", "post_birth_sample_flag"]

meta = {}
samples = {}
for t in TIMEPOINTS:
    temp = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/MTBL/plasma/MTBL_plasma_T{t}.csv")
    mask = np.where(pd.isna(temp).sum(axis=0) > 0.5*len(temp.columns), True, False)
    print("Dropped analytes for >50% missing: " )
    print([x for x in temp.loc[:, mask].columns])
    temp = temp.loc[:, ~mask]

    mask = np.where(pd.isna(temp).sum(axis=1) > 0.2*len(temp.index), True, False)
    print("Dropped samples for >20% missing: " )
    print([x for x in temp.loc[mask, "SampleID"]])
    temp = temp.loc[~mask,:]

    meta[t] = temp[[x for x in meta_cols if x in temp.columns]]
    expr_temp = temp.drop([x for x in meta_cols if x in temp.columns], axis=1)
    imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
    samples[t]  = pd.DataFrame(imputer.fit_transform(expr_temp))
    samples[t].index = temp["SampleID"]
    samples[t].columns = expr_temp.columns



unique_samples = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True)
unique_samples = unique_samples[unique_samples["Group"] != "Control"]
bySample = unique_samples[[x for x in meta_cols if x in temp.columns]].set_index("SampleID").to_dict('index')

# get subject-level information -> dict
bySubject = unique_samples.groupby('SubjectID').agg(lambda x: x.unique().tolist()).reset_index().set_index("SubjectID").to_dict('index')


temp = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/MTBL/placenta/MTBL_placenta_cleaned_with_metadata.csv")
mask = np.where(pd.isna(temp).sum(axis=0) > 0.5*len(temp.columns), True, False)
print("Dropped analytes for >50% missing: " )
print([x for x in temp.loc[:, mask].columns])
temp = temp.loc[:, ~mask]

mask = np.where(pd.isna(temp).sum(axis=1) > 0.2*len(temp.index), True, False)
print("Dropped samples for >20% missing: " )
print([x for x in temp.loc[mask, "SampleID"]])
temp = temp.loc[~mask,:]

meta_placenta = temp[[x for x in meta_cols if x in temp.columns]]
expr_temp = temp.drop([x for x in meta_cols if x in temp.columns], axis=1)
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
samples_placenta  = pd.DataFrame(imputer.fit_transform(expr_temp))
samples_placenta.index = temp["SampleID"]
samples_placenta.columns = expr_temp.columns


print(f"Total Samples: {len(meta_placenta.index)}")
print(f"Num. of Unique Subjects: {len(meta_placenta["SubjectID"].unique())}")

print(f"Total Features = {len(list(set(samples_placenta.columns)))}")


In [116]:
temp = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/MTBL/plasma/MTBL_plasma_cleaned_with_metadata.csv")

In [117]:
meta_cols = [ "SubjectID", "Batch", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint", "SampleTimepoint",
             "MetadataCanonicalID", "labor_onset", "indicated_onset", "spont_labor_flag", "indicated_onset_flag", "cat_labor_onset_flag",
             "within_0_1wk_delivery_flag", "post_birth_sample_flag"]
temp = temp.drop([x for x in temp.columns if x in meta_cols], axis=1)

In [118]:
temp.index[temp.isna().sum(axis=1) >0]


RangeIndex(start=0, stop=0, step=1)

In [120]:
temp


,SampleID,Glycyl-Threonine_POS,L-Glutamine_POS,Cysteineglutathione disulfide_POS,8-Cyano-fluoroquinolone_POS,N-methyl-4-nitrotryptophan_POS,[Similar to: (4-benzoylpiperidino)[4-(tert-butyl)phenyl]methanone; ΔMass: 38.0426 Da]_POS,NP-015114_POS,Diaminopimelic acid_POS,Doxylamine_POS,...,unk_n11_NEG_273.9895_RT10.40,unk_n9_NEG_273.9896_RT11.04,unk_n682_NEG_345.0726_RT11.05,unk_n441_NEG_525.3312_RT11.15,unk_n189_NEG_228.9940_RT11.33,unk_n449_NEG_525.3312_RT11.55,unk_n146_NEG_228.9939_RT11.62,unk_n454_NEG_525.3312_RT11.94,unk_n219_NEG_116.9285_RT11.96,unk_n2019_NEG_382.1452_RT12.08
0,DP3-0005A,8.346328,2.514693,0.167109,0.031191,0.351870,0.067459,0.177146,0.210556,0.017678,...,6.394878,6.673255,0.193467,1.227990,1.324445,1.146580,1.998351,1.124812,1.727648,-0.041024
1,DP3-0005B,7.056433,2.107925,0.143025,0.011596,0.355536,1.154299,0.176231,0.175763,0.018016,...,6.315891,6.588021,0.167462,1.171892,1.410654,1.117089,1.274740,1.140697,1.764429,-0.027348
2,DP3-0005C,7.299477,2.387536,0.096931,-0.039117,0.437729,1.319555,0.179498,0.122342,0.021437,...,6.145471,6.600887,0.148170,1.177536,1.269878,1.104243,1.133663,1.126799,1.882792,-0.030498
3,DP3-0005D,6.639975,2.244456,0.075455,-0.043117,0.289928,0.983528,0.176852,0.063075,0.023020,...,3.233523,3.148372,0.104398,1.047415,1.395925,1.146115,2.180362,1.136948,1.173935,-0.016294
4,DP3-0006A,7.268753,2.394954,0.055455,0.239356,0.389954,1.201859,0.101048,0.172390,0.024529,...,6.495568,6.619125,0.097366,0.994810,1.862163,0.993501,1.822578,0.979016,1.625077,0.018030
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
519,DP3-0423EC,6.845868,1.673613,0.019029,0.032224,-0.015299,1.007905,-0.050359,-0.004030,0.003519,...,6.007902,5.638348,0.013462,1.144041,1.279078,1.096739,1.308890,1.089795,1.531953,0.193536
520,DP3-0423ED,6.328161,1.967214,0.044449,0.028749,-0.021252,0.989271,-0.068263,0.065054,0.000237,...,5.225746,7.206120,0.014694,0.831156,0.891250,0.837946,0.911411,0.840984,1.681527,0.019270
521,DP3-0423EE,6.387140,1.961225,0.047956,0.142375,0.284955,0.985092,-0.066685,-0.004550,0.003281,...,5.803492,5.525013,0.015416,1.132153,1.108594,1.077742,1.633262,1.071372,1.495478,0.110952
522,DP3-0436A,7.141014,1.908143,0.039461,0.268817,0.370094,1.022136,-0.069954,0.148268,0.012542,...,6.238673,5.685225,-0.018593,1.273903,2.388977,1.252529,2.427505,1.253774,2.054429,0.000991
